# Maintenance

This notebook covers operational maintenance tasks for semantic models, including refresh management, TOM updates, RLS validation, and model hygiene activities.

In [ ]:
# Parameters
workspace_name = "Demo"
dataset_name = "card_udf"
rls_workspace_name = "Demo"
rls_user_1 = "Jai@enterprisedatastrategies.com"
rls_user_2 = "edward@enterprisedatastrategies.com"

In [ ]:
# Imports
# Imports used by demo(s): Real world skills - Parsing SQL Tables Used; Real world skills - Move Measures into Folder
import pandas as pd
# Imports used by demo(s): Real world skills - Parsing SQL Tables Used
import re
# Imports used by demo(s): General
import sempy.fabric
# Imports used by demo(s): Semantic Model Refresh Operations; How to Refresh a specific Table in a Semantic Model; How to fetch refresh history; How to enable/disable a refresh; Real World:; How to update a Semantic Model Refresh Schedule; Semantic Link Labs the TOM Package; Basic Measure Operations; Format DAX; Add a Measure; Update a Measure; Calculation Groups; Add a calculation group item; Update a calculaton group item; Real world skills - Parsing SQL Tables Used; User 1; User 2; Real world skills - Move Measures into Folder
import sempy_labs as labs

## Semantic Model Refresh Operations
##### How to Refresh a Full Semantic Model 

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.html#sempy_labs.refresh_semantic_model

# Refreshes a full Semantic Model
labs.refresh_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name
)

##### How to Refresh a specific Table in a Semantic Model 

*Real World: This is good for refreshing a specific table like a RLS Table*

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.html#sempy_labs.refresh_semantic_model

# How to refresh a specific table or list of tables
labs.refresh_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    tables=["Fact Table"]
)

##### How to fetch refresh history

*Real World: Trying to extract why a Semantic Model is consistently failing*

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.html#sempy_labs.get_semantic_model_refresh_history

# How to fetch a semantic model's refresh history
labs.get_semantic_model_refresh_history(
    dataset=dataset_name,
    workspace=workspace_name
)

##### How to enable/disable a refresh

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.html#sempy_labs.enable_semantic_model_scheduled_refresh

# How to enable or disable a scheduled refresh
labs.enable_semantic_model_scheduled_refresh(
    dataset=dataset_name,
    workspace=workspace_name,
    enable=True
)

*Real World:*
- *You need to shut down a bunch of Semantic Models for a database update*
- *Mass failure event and a bunch of Semantic Models killed thier refresh schedule*

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.html#sempy_labs.enable_semantic_model_scheduled_refresh

# workspace
workspace = workspace_name

# list of Semantic Models
semantic_models = [
    "card_udf",
    "Custom Bar Chart with DAX UDFs",
    "Sample Dataset One"
]

# enable or disable
enable_refresh = False

# for each semantic model loop through and enable or disable
for model in semantic_models:
    labs.enable_semantic_model_scheduled_refresh(
        dataset=model,
        workspace=workspace,
        enable=enable_refresh
    )

##### How to update a Semantic Model Refresh Schedule
*Real World: A process got pushed back and you need to adjust a bunch of scheduled refreshes*

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.html#sempy_labs.update_semantic_model_refresh_schedule

# update a Refresh schedule for a semantic model
labs.update_semantic_model_refresh_schedule(
    dataset=dataset_name,
    workspace=workspace_name,
    days=["Monday", "Wednesday", "Friday"],
    times=["06:00", "18:00"],
    time_zone="Eastern Standard Time"
)

## Semantic Link Labs the TOM Package
##### How to Open/Close a Connection to a Semantic Model

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper

# Open a TOM Connection
tom = labs.tom.TOMWrapper(
    dataset=dataset_name,
    workspace=workspace_name,
    readonly=True #False
)

try:
    print(tom.model.Database.Name)
finally:
    # Closing a Tom Connection
    tom.close()

##### Basic Measure Operations
List All Measures

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.all_measures

# Do not need a close here because within the With block it closes in a finally
with labs.tom.connect_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name
) as tom:
    for measure in tom.all_measures():
        print(f"{measure.Parent.Name}.{measure.Name}")
    tom.close()

Format DAX

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.format_dax

# Format all DAX
with labs.tom.connect_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    readonly=False   # must be False to save changes
) as tom:
    tom.format_dax()

In [ ]:
from sempy_labs.tom import _model as _tom_model

_tom_model.TOMWrapper._dax_formatting = {
    "measures": [],
    "calculated_columns": [],
    "calculated_tables": [],
    "calculation_items": [],
    "rls": [],
}

print("State reset complete")

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.all_measures
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.format_dax

# Format a single measure
with labs.tom.connect_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    readonly=False
) as tom:
    measure = next(m for m in tom.all_measures() if m.Name == "Chip 2")
    tom.format_dax(object=measure)

In [ ]:
from sempy_labs.tom import _model as _tom_model

_tom_model.TOMWrapper._dax_formatting = {
    "measures": [],
    "calculated_columns": [],
    "calculated_tables": [],
    "calculation_items": [],
    "rls": [],
}

print("State reset complete")

Add a Measure

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.add_measure

# Add a measure to the semantic model
with labs.tom.connect_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    readonly=False
) as tom:
    tom.add_measure(
        table_name="Fact Table",
        measure_name="Total Sales New",
        expression="SUM('Fact Table'[order_val])",
        format_string="#,0.00",
        description="Sum of all sales",
        display_folder="Revenue"
    )

Update a Measure

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.update_measure

# Update a measure — only the fields you pass will change
# https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.update_measure
with labs.tom.connect_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    readonly=False
) as tom:
    tom.update_measure(
        measure_name="Total Sales",
        display_folder="Revenue2"
    )

##### Calculation Groups
Adding a calculation group 

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.add_calculation_group
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.add_calculation_item

# Add a calculation group with calculation items
with labs.tom.connect_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    readonly=False
) as tom:
    # Step 1: Add the calculation group table
    tom.add_calculation_group(
        name="Time Intelligence",
        precedence=1,
        column_name="Name"       # the column users see in the slicer
    )

    # Step 2: Add calculation items to it
    tom.add_calculation_item(
        table_name="Time Intelligence",
        calculation_item_name="Current",
        expression="SELECTEDMEASURE()",
        ordinal=0
    )
    tom.add_calculation_item(
        table_name="Time Intelligence",
        calculation_item_name="YTD",
        expression="CALCULATE(SELECTEDMEASURE(), DATESYTD('Dim Date'[date_actual]))",
        ordinal=1
    )
    tom.add_calculation_item(
        table_name="Time Intelligence",
        calculation_item_name="PY",
        expression="CALCULATE(SELECTEDMEASURE(), SAMEPERIODLASTYEAR('Dim Date'[date_actual]))",
        ordinal=2
    )

Add a calculation group item

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.add_calculation_item

# Add a single calculation item to an existing calculation group
with labs.tom.connect_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    readonly=False
) as tom:
    tom.add_calculation_item(
        table_name="Time Intelligence",      # existing calc group table name
        calculation_item_name="MTD",
        expression="CALCULATE(SELECTEDMEASURE(), DATESMTD(CALENDAR[Date]))",
        ordinal=3,
        format_string_expression='"$#,0.00"'   # optional — dynamic format string
    )

Update a calculaton group item

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.html#sempy_labs.refresh_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.TOMWrapper.update_calculation_item

"""
Update an existing calculation item — only fields you pass will change
Might need to manually refresh the calculation group after updating
a calculation item to ensure changes are reflected immediately.
"""
with labs.tom.connect_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    readonly=False
) as tom:
    tom.update_calculation_item(
        table_name="Time Intelligence",
        calculation_item_name="MTD",
        expression="CALCULATE(SELECTEDMEASURE(), DATESMTD('Dim Date'[date_actual]))",
        ordinal=4
    )

"""
Optional: refresh only the calculation group table.

labs.refresh_semantic_model(
    dataset=dataset_name,
    workspace=workspace_name,
    tables=["Time Intelligence"]
)
"""

## Real world skills - Parsing SQL Tables Used
##### Regex

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.tom.html#sempy_labs.tom.connect_semantic_model

rows = []
pattern = re.compile(r'(?i)\b(from|join|left\s+join|right\s+join|inner\s+join|full\s+join|cross\s+join)\s+([^\s,;\)\(]+)')

with labs.tom.connect_semantic_model(dataset=dataset_name, workspace=workspace_name, readonly=True) as tom:
    for t in tom.model.Tables:
        for p in t.Partitions:
            expr = getattr(getattr(p, "Source", None), "Expression", None)
            if not isinstance(expr, str):
                continue

            for criteria, obj in pattern.findall(expr):
                # Clean common SQL formatting noise
                obj = obj.strip().strip("[]").replace("[", "").replace("]", "")
                rows.append({
                    "PowerBI_Table_Name": t.Name,
                    "Criteria_Type": " ".join(criteria.upper().split()),
                    "Schema_Database_Table": obj
                })
    tom.close()

df_sql_refs = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
df_sql_refs

## Real world skills - RLS Check

##### User 1

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.html#sempy_labs.evaluate_dax_impersonation

dataset = dataset_name
workspace = rls_workspace_name
run_as = rls_user_1

dax_query = """
EVALUATE
ROW("Row Count", COUNTROWS('Fact Table'))
"""

labs.evaluate_dax_impersonation(
    dataset=dataset,
    workspace=workspace,
    dax_query=dax_query,
    user_name=run_as
)

##### User 2

In [ ]:
# Semantic Link Labs docs reference
# - https://semantic-link-labs.readthedocs.io/en/latest/sempy_labs.html#sempy_labs.evaluate_dax_impersonation

dataset = dataset_name
workspace = rls_workspace_name
run_as = rls_user_2

dax_query = """
EVALUATE
ROW("Row Count", COUNTROWS('Fact Table'))
"""

labs.evaluate_dax_impersonation(
    dataset=dataset,
    workspace=workspace,
    dax_query=dax_query,
    user_name=run_as
)

## Real world skills - Move Measures into Folder
##### Move Measures into a Folder

In [ ]:
import pandas as pd

dataset=dataset_name
workspace=workspace_name
measures_table = "KPI"

with labs.tom.connect_semantic_model(dataset=dataset, workspace=workspace, readonly=False) as tom:
    if measures_table not in [t.Name for t in tom.model.Tables]:
        tom.add_calculated_table(
            name=measures_table,
            expression='ROW("Measure Holder", BLANK())',
            hidden=False,
        )

        new_table = tom.model.Tables[measures_table]
        if new_table.Columns.Count > 0:
            new_table.Columns[0].IsHidden = True

    moves = []
    for m in tom.all_measures():
        if m.Parent.Name == measures_table:
            continue

        moves.append(
            {
                "old_table": m.Parent.Name,
                "name": m.Name,
                "expression": m.Expression,
                "format_string": getattr(m, "FormatString", None),
                "hidden": getattr(m, "IsHidden", False),
                "description": getattr(m, "Description", None),
                "display_folder": getattr(m, "DisplayFolder", None),
                "data_category": getattr(m, "DataCategory", None),
                "format_string_expression": (
                    m.FormatStringDefinition.Expression
                    if getattr(m, "FormatStringDefinition", None) is not None
                    else None
                ),
            }
        )

    for measure in moves:
        src_table = tom.model.Tables[measure["old_table"]]
        target_table = tom.model.Tables[measures_table]

        # Add only if target does not already have it
        if not any(m.Name == measure["name"] for m in target_table.Measures):
            tom.add_measure(
                table_name=measures_table,
                measure_name=measure["name"],
                expression=measure["expression"],
                format_string=measure["format_string"],
                hidden=measure["hidden"],
                description=measure["description"],
                display_folder=measure["display_folder"],
                format_string_expression=measure["format_string_expression"],
                data_category=measure["data_category"],
            )

        # Remove only if source still has it, and remove by object reference
        src_measure = next((m for m in src_table.Measures if m.Name == measure["name"]), None)
        if src_measure is not None:
            src_table.Measures.Remove(src_measure)
        

    result = pd.DataFrame(
        [
            {"Table": m.Parent.Name, "Measure": m.Name}
            for m in tom.all_measures()
        ]
    )

result